__Importing libraries__

In [1]:
import pandas as pd 
import sqlite3
import numpy as np
import random
from faker import Faker
import re
import os

In [2]:
from datetime import datetime, timedelta


from faker import Faker
fake=Faker()

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)


In [4]:
random.seed(42)
np.random.seed(42)

##  Step1:Data Generation

In [5]:
NUM_CUSTOMERS = int(input("Enter number of customers: "))

NUM_PRODUCTS = int(input("Enter number of products: "))

NUM_ORDERS = int(input("Enter number of orders: "))

NUM_ORDER_ITEMS= int(input("Enter number of items:"))

Enter number of customers:  5000
Enter number of products:  2000
Enter number of orders:  366
Enter number of items: 378


In [6]:

# CUSTOMERS

def customer_data():
    customer_types = ["REGULAR", "PREMIUM", "VIP"]
    customer_type_weights = [0.70, 0.20, 0.10]
    
    customers = []

    for customer_id in range(1, NUM_CUSTOMERS + 1):
    
        email = fake.email()
    
        # 3% invalid emails
        if random.random() < 0.03:
            invalid_type = random.choice([1, 2, 3])
    
            if invalid_type == 1:
                email = email.replace("@", "")
            elif invalid_type == 2:
                email = email.split("@")[0] + "@"
            else:
                email = email.split("@")[0]
    
        registration_date = fake.date_between(
            start_date="-5y",
            end_date="today")
    
        customers.append({
            "customer_id": customer_id,
            "customer_name": fake.name(),
            "email": email,
            "registration_date": registration_date,
            "customer_type": random.choices(
                customer_types,
                weights=customer_type_weights,
                k=1
            )[0]
        })
    
    customers_df = pd.DataFrame(customers)
    return customers_df
customers_df = customer_data()

# PRODUCTS

def products_data():
    category_map = {
        "Electronics": ["Mobile","Laptop","Headphones","Camera","Tablet"],
        "Clothing": ["Shirt","Jeans","Shoes","Jacket","T-Shirt"],
        "Home": ["Furniture","Kitchen","Decor","Lighting","Storage"],
        "Books": ["Fiction","Education","Comics","Biography","Science"]
    }
    
    products = []
    
    for product_id in range(1, NUM_PRODUCTS + 1):
    
        category = random.choice(list(category_map.keys()))
        subcategory = random.choice(category_map[category])
    
        product_name = f"{fake.word().title()} {subcategory}"
    
        # Product quality issues
        if random.random() < 0.05:
            style = random.choice([1, 2, 3])
    
            if style == 1:
                product_name = f"   {product_name}   "
            elif style == 2:
                product_name = product_name.upper()
            else:
                product_name = product_name.swapcase()
    
        products.append({
            "product_id": product_id,
            "product_name": product_name,
            "category": category,
            "subcategory": subcategory,
            "cost_price": round(random.uniform(5, 1500), 2)
        })
    
    products_df = pd.DataFrame(products)
    return products_df
products_df=products_data()


# ORDERS
def order_data():
    statuses = ["PLACED","SHIPPED","DELIVERED", "CANCELLED","RETURNED"]
    
    status_weights = [0.10,0.20,0.60,0.05,0.05]
    
    orders = []
    
    for order_id in range(1, NUM_ORDERS + 1):
    
        # 5% missing customer IDs
        if random.random() < 0.05:
            customer_id = None
        else:
            customer_id = random.randint(1, NUM_CUSTOMERS)
    
        order_datetime = fake.date_time_between(
            start_date="-2y",
            end_date="now")
        
        discount_percent = round
        (random.uniform(0, 100),2
        )

    
        # Some wrong date formats
        if random.random() < 0.03:
            order_date = order_datetime.strftime("%d-%m-%Y")
        else:
            order_date = order_datetime.strftime("%Y-%m-%d %H:%M:%S")

            
        orders.append({
            "order_id": order_id,
            "customer_id": customer_id,
            "order_date": order_date,
            "status": random.choices(
                statuses,
                weights=status_weights,
                k=1
            )[0],
            "region_code":random.choice(["NORTH", "SOUTH", "EAST", "WEST"])
        })
            
    orders_df = pd.DataFrame(orders)

    return orders_df
orders_df=order_data()

# ORDER ITEMS

def order_items_data(orders_df, products_df):

    order_ids = orders_df["order_id"].tolist()
    product_ids = products_df["product_id"].tolist()

    order_items = []

    for item_id in range(1, NUM_ORDER_ITEMS + 1):

        quantity = random.randint(1, 10)

        # 3% negative quantity
        if random.random() < 0.03:
            quantity = -quantity

        order_items.append({
            "item_id": item_id,
            "order_id": random.choice(order_ids),   # always valid
            "product_id": random.choice(product_ids),  # always valid
            "quantity": quantity,
            "unit_price": round(random.uniform(10, 2500), 2),
            "discount_percent": round(random.uniform(0, 100), 2)
        })

    return pd.DataFrame(order_items)
order_items_df=order_items_data(orders_df,products_df)


__Locating Path__

In [7]:
DATA_DIR = os.path.join(os.getcwd(), "data", "raw_data")
os.makedirs(DATA_DIR, exist_ok=True)

def save_csvs(customers_df, products_df, orders_df, order_items_df):
    """Save all generated DataFrames as CSV files to DATA_DIR."""
    customers_df.to_csv(os.path.join(DATA_DIR, "customers.csv"),    index=False)
    products_df.to_csv(os.path.join(DATA_DIR, "products.csv"),      index=False)
    orders_df.to_csv(os.path.join(DATA_DIR, "orders.csv"),          index=False)
    order_items_df.to_csv(os.path.join(DATA_DIR, "order_items.csv"),index=False)
    logging.info(f"CSV files saved to: {DATA_DIR}")




__Loading Data into CSV__

In [8]:
def generate_data():
    customers_df = customer_data()
    products_df = products_data()
    orders_df = order_data()
    order_items_df =order_items_data(orders_df, products_df)

    save_csvs(
        customers_df,
        products_df,
        orders_df,
        order_items_df
    )

generate_data()    

2026-07-12 22:01:47,895 | INFO | CSV files saved to: /Users/jain/Celebal_Internship_Assignment/E_Commerece_Order_Analytics/data/raw_data


In [10]:
def check_referential_integrity(
        order_items_df,
        orders_df,
        products_df):


    invalid_orders = order_items_df[
        ~order_items_df["order_id"]
        .isin(orders_df["order_id"])
    ]


    invalid_products = order_items_df[
        ~order_items_df["product_id"]
        .isin(products_df["product_id"])
    ]


    print(
        f"Invalid Order IDs   : {len(invalid_orders)}"
    )

    print(
        f"Invalid Product IDs : {len(invalid_products)}"
    )


    return invalid_orders, invalid_products

In [11]:
tables = {
    "Customers": customers_df,
    "Products": products_df,
    "Orders": orders_df,
    "Order Items": order_items_df
}


In [12]:
logging.info("Raw Data Summary")

2026-07-12 22:01:47,905 | INFO | Raw Data Summary


__Summary Of Raw Data__

In [13]:
def data_summary(tables):
    
    def data_check_customers(customers_df):

        invalid_emails = customers_df[
            ~customers_df["email"].str.match(
                r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
                na=False
            )
        ]
    
        print(f"Missing values   :{customers_df.isnull().sum().sum()}")
        print(f"Invalid Emails   : {len(invalid_emails)}")


    def data_check_products(products_df):
    
        extra_spaces = (
            products_df["product_name"] !=
            products_df["product_name"].str.strip()
        ).sum()
    
        uppercase = (
            products_df["product_name"] ==
            products_df["product_name"].str.upper()
        ).sum()
    
        mixed_case = (
            products_df["product_name"] != products_df["product_name"].str.title()
        ).sum()    
        print(f"Extra Spaces     : {extra_spaces}")
        print(f"Upper Case       : {uppercase}")
        print(f"Mixed Case       : {mixed_case}")


    def data_check_order(orders_df):
    
        duplicate_orders = orders_df[
            orders_df["order_id"].duplicated()
        ]
    
        missing_order_id = orders_df[
            orders_df["order_id"].isna()
        ]
    
        print(f"No of duplicate order :{ len(duplicate_orders)}")
        print(f"No of missing order   :{len(missing_order_id)}")
        


    def data_check_order_items(order_items_df):
    
        negative_qty = order_items_df[
            order_items_df["quantity"] < 0
        ]
    
        invalid_orders = order_items_df[
            ~order_items_df["order_id"].isin(orders_df["order_id"])
        ]
    
        print(f"Negative Quantity : {len(negative_qty)}")
        print(f"Invalid Order IDs : {len(invalid_orders)}")


            
            
        
    for name, df in tables.items():
        print(f"Table: {name}")
        print(f"No. of rows      : {df.shape[0]}")
        print(f"Duplicate data   : {df.duplicated().sum()}")
        print(f"Empty values     : {df.isna().sum().sum()}")
    
        
        if name=='Customers':
            data_check_customers(df)
    
            print("--**--")
    
        elif name =='Products':
            data_check_products(df)
    
            print("--**--")
    
        elif name=='Orders':
            data_check_order(df)
    
            print("--**--")
    
        else:
            data_check_order_items(df)

    

data_summary(tables)

Table: Customers
No. of rows      : 5000
Duplicate data   : 0
Empty values     : 0
Missing values   :0
Invalid Emails   : 153
--**--
Table: Products
No. of rows      : 2000
Duplicate data   : 0
Empty values     : 0
Extra Spaces     : 33
Upper Case       : 40
Mixed Case       : 73
--**--
Table: Orders
No. of rows      : 366
Duplicate data   : 0
Empty values     : 20
No of duplicate order :0
No of missing order   :0
--**--
Table: Order Items
No. of rows      : 378
Duplicate data   : 0
Empty values     : 0
Negative Quantity : 12
Invalid Order IDs : 0


In [14]:
check_referential_integrity(
    order_items_df,
    orders_df,
    products_df
)

Invalid Order IDs   : 0
Invalid Product IDs : 0


(Empty DataFrame
 Columns: [item_id, order_id, product_id, quantity, unit_price, discount_percent]
 Index: [],
 Empty DataFrame
 Columns: [item_id, order_id, product_id, quantity, unit_price, discount_percent]
 Index: [])

## Step 2 : Data Cleaning

In [15]:
from pandas.api.types import is_numeric_dtype

# Output folder
CLEANED_DIR = os.path.join(os.getcwd(), "data", "cleaned_data")
os.makedirs(CLEANED_DIR, exist_ok=True)


def cleaning_data(tables, output_folder=CLEANED_DIR):
   
    os.makedirs(output_folder, exist_ok=True)

    cleaned_tables = {}

    today = pd.Timestamp.today().normalize()

    for name, df in tables.items():

        print(f"\nCleaning {name}...")

        original_rows = len(df)

        # Remove duplicates

        df = df.drop_duplicates()

        #
        # Remove completely empty rows
        df = df.dropna(how="all")

        # Clean text columns
        for col in df.select_dtypes(include=["object", "string"]).columns:

            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .replace("", pd.NA)
            )

        # Convert dates
        for col in df.columns:
            if "date" in col.lower():

                df[col] = pd.to_datetime(
                    df[col],
                    errors="coerce"
                )

                # Remove future dates
                df = df[df[col] <= today]

        # Convert numeric columns
        numeric_columns = [
            "quantity",
            "unit_price",
            "cost_price",
            "total_amount",
            "discount_percent"
        ]

        for col in numeric_columns:

            if col in df.columns:

                df[col] = pd.to_numeric(
                    df[col],
                    errors="coerce"
                )

        # Check discount_percent
        if "discount_percent" in df.columns:

            df["discount_percent"] = (
                df["discount_percent"]
                .fillna(0)
            )

            # Flag invalid discounts
            df["invalid_discount"] = (
                (df["discount_percent"] < 0) |
                (df["discount_percent"] > 100)
            )

            # Keep values within 0-100
            df["discount_percent"] = (
                df["discount_percent"]
                .clip(0, 100)
            )

        # Quantity Checking
        if "quantity" in df.columns:

            df["quantity"] = (
                df["quantity"]
                .fillna(0)
            )

            # Returns
            df["is_return"] = df["quantity"] < 0

            # Remove negative quantity rows
            df = df[df["quantity"] >= 0]

        # Email validation
        if "email" in df.columns:

            df = df[
                df["email"]
                .astype("string")
                .str.match(
                    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
                    na=False
                )
            ]

        # Remove missing Order IDs
        if "order_id" in df.columns:

            df = df[
                df["order_id"].notna()
            ]

        # Standardize product names
        # 
        if (
            name.lower() == "products"
            and "product_name" in df.columns
        ):

            df["product_name"] = (
                df["product_name"]
                .str.title()
            )

        # Remove negative prices
        
        for col in [
            "unit_price",
            "cost_price",
            "total_amount"
        ]:

            if col in df.columns:

                df = df[
                    df[col] >= 0
                ]

        # Remove rows with missing values
        
        df = df.dropna()

        # Save cleaned table
        cleaned_tables[name] = df

        output_file = os.path.join(
            output_folder,
            f"{name.lower().replace(' ', '_')}_cleaned.csv"
        )

        df.to_csv(output_file, index=False)

        logging.info(
            f"{name}: {original_rows} -> {len(df)} rows"
        )

        print(
            f"{name}: {original_rows} -> {len(df)} rows"
        )

        print(f"Saved: {output_file}")

    print("\nCleaning completed successfully.")

    return cleaned_tables

# Run cleaning

cleaned_tables = cleaning_data(tables)

logging.info("All tables cleaned successfully.")

2026-07-12 22:01:47,953 | INFO | Customers: 5000 -> 4847 rows
2026-07-12 22:01:47,960 | INFO | Products: 2000 -> 2000 rows
2026-07-12 22:01:47,965 | INFO | Orders: 366 -> 328 rows
2026-07-12 22:01:47,969 | INFO | Order Items: 378 -> 366 rows
2026-07-12 22:01:47,969 | INFO | All tables cleaned successfully.



Cleaning Customers...
Customers: 5000 -> 4847 rows
Saved: /Users/jain/Celebal_Internship_Assignment/E_Commerece_Order_Analytics/data/cleaned_data/customers_cleaned.csv

Cleaning Products...
Products: 2000 -> 2000 rows
Saved: /Users/jain/Celebal_Internship_Assignment/E_Commerece_Order_Analytics/data/cleaned_data/products_cleaned.csv

Cleaning Orders...
Orders: 366 -> 328 rows
Saved: /Users/jain/Celebal_Internship_Assignment/E_Commerece_Order_Analytics/data/cleaned_data/orders_cleaned.csv

Cleaning Order Items...
Order Items: 378 -> 366 rows
Saved: /Users/jain/Celebal_Internship_Assignment/E_Commerece_Order_Analytics/data/cleaned_data/order_items_cleaned.csv

Cleaning completed successfully.


__Summary Of Cleaned/Processed Data__

In [16]:
data_summary(cleaned_tables)

Table: Customers
No. of rows      : 4847
Duplicate data   : 0
Empty values     : 0
Missing values   :0
Invalid Emails   : 0
--**--
Table: Products
No. of rows      : 2000
Duplicate data   : 0
Empty values     : 0
Extra Spaces     : 0
Upper Case       : 0
Mixed Case       : 0
--**--
Table: Orders
No. of rows      : 328
Duplicate data   : 0
Empty values     : 0
No of duplicate order :0
No of missing order   :0
--**--
Table: Order Items
No. of rows      : 366
Duplicate data   : 0
Empty values     : 0
Negative Quantity : 0
Invalid Order IDs : 0


## Step 3 :Data Analysis using SQL

__Connection to SQL__

In [17]:
import sqlite3

In [18]:

def get_connection(db_name: str = "ecommerce_analysis.db") -> sqlite3.Connection:
    """Create and return a database connection."""
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

conn = get_connection()
cursor = conn.cursor()
print(f'Database Created Successfully')

customers_df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

products_df.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

orders_df.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

order_items_df.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded successfully")

Database Created Successfully
Data loaded successfully


__Total revenue per category (revenue = quantity × unit_price × (1 - discount_percent/100))__

In [19]:
pd.read_sql("""SELECT p.category,SUM(oi.quantity *oi.unit_price *(1 - oi.discount_percent / 100.0)) AS total_revenue

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

GROUP BY p.category

ORDER BY total_revenue DESC""",conn)

,category,total_revenue
0,Clothing,362437.706857
1,Electronics,329886.773453
2,Books,283942.450069
3,Home,233128.292532


__Top 10 customers by total order value__

In [20]:
pd.read_sql("""SELECT
    c.customer_id,
    c.customer_name,

    SUM(
        oi.quantity *
        oi.unit_price *
        (1 - oi.discount_percent / 100.0)
    ) AS total_order_value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY
    c.customer_id,
    c.customer_name

ORDER BY total_order_value DESC

LIMIT 10""",conn)

,customer_id,customer_name,total_order_value
0,2693,Melissa Black,30227.572410
1,2532,Amy Hayes,23145.391494
2,2117,Robert Rice,22196.889630
3,2826,Danielle Ramos,21513.638586
4,4877,Annette Burke,21400.974748
5,4549,Kevin Sanders,20809.175450
6,1292,Ruth Hess,20632.124856
7,1400,Samantha Delacruz,20263.291580
8,2404,Nicholas Dominguez,17440.567984
9,1008,Mark Lindsey,17404.690176


__Month-wise order count for the last 12 months__

In [21]:
pd.read_sql("""SELECT

strftime('%Y-%m', order_date) as month,

COUNT(order_id) AS order_count

FROM orders

WHERE order_date >= date(
    'now',
    '-12 months'
)

GROUP BY month

ORDER BY month""",conn)

,month,order_count
0,None,6
1,2025-07,12
2,2025-08,12
3,2025-09,17
4,2025-10,8
5,2025-11,15
6,2025-12,13
7,2026-01,15
8,2026-02,12
9,2026-03,8


__Find customers who placed orders but never had any item delivered__

In [22]:
pd.read_sql("""SELECT DISTINCT c.customer_id, c.customer_name

FROM customers c

JOIN orders o

ON c.customer_id = o.customer_id

WHERE c.customer_id NOT IN (

    SELECT customer_id

    FROM orders

    WHERE status = 'DELIVERED')

""",conn)

,customer_id,customer_name


__Products that were ordered but had more returns than purchases__

In [23]:
pd.read_sql("""SELECT

p.product_id,
p.product_name,


SUM(
    CASE
        WHEN oi.quantity > 0
        THEN oi.quantity
        ELSE 0
    END
) AS purchases,


ABS(
    SUM(
        CASE
            WHEN oi.quantity < 0
            THEN oi.quantity
            ELSE 0
        END
    )
) AS returns


FROM products p


JOIN order_items oi

ON p.product_id = oi.product_id


GROUP BY
p.product_id,
p.product_name


HAVING returns > purchases""",conn)

,product_id,product_name,purchases,returns
0,515,Inside Camera,0,3
1,531,Century Science,0,2
2,594,Land Lighting,0,1
3,659,Executive Camera,0,9
4,667,Follow Storage,0,6
5,741,Culture Decor,0,8
6,923,Practice Tablet,0,9
7,1206,Fish Jacket,0,9
8,1224,Building Kitchen,0,6
9,1227,Student Shirt,0,8


__Calculate the return rate (returned items / total items) per category__

In [24]:
pd.read_sql("""SELECT

p.category,


ROUND(

SUM(
    CASE
        WHEN oi.quantity < 0
        THEN ABS(oi.quantity)
        ELSE 0
    END
)

/

SUM(
    ABS(oi.quantity)
) * 100,

2

) AS return_rate_percentage


FROM products p


JOIN order_items oi

ON p.product_id = oi.product_id


GROUP BY p.category""",conn)

,category,return_rate_percentage
0,Books,0.0
1,Clothing,0.0
2,Electronics,0.0
3,Home,0.0


__Calculate running total of revenue per region, ordered by date__

In [25]:
pd.read_sql("""WITH daily_sales AS (

    SELECT
        o.region_code,
        DATE(o.order_date) AS order_date,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS daily_revenue

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    GROUP BY
        o.region_code,
        DATE(o.order_date)

)

SELECT

    region_code,
    order_date,
    daily_revenue,

    SUM(daily_revenue) OVER(
        PARTITION BY region_code
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total


FROM daily_sales

ORDER BY
    region_code,
    order_date""",conn)

,region_code,order_date,daily_revenue,running_total
0,EAST,None,1022.783840,1022.783840
1,EAST,2024-07-22,4948.757635,5971.541475
2,EAST,2024-09-03,11331.207786,17302.749261
3,EAST,2024-09-22,308.333813,17611.083074
4,EAST,2024-09-24,623.440940,18234.524014
...,...,...,...,...
203,WEST,2026-05-25,2547.304944,288474.829364
204,WEST,2026-05-26,9834.823440,298309.652804
205,WEST,2026-06-08,30227.572410,328537.225214
206,WEST,2026-06-29,426.960000,328964.185214


__For each category, rank products by total revenue.__

In [26]:
pd.read_sql("""WITH product_revenue AS (

    SELECT
        p.category,
        p.product_name,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS total_revenue

    FROM products p

    JOIN order_items oi
    ON p.product_id = oi.product_id

    GROUP BY
        p.category,
        p.product_name

)

SELECT

    category,
    product_name,
    total_revenue,

    DENSE_RANK() OVER(
        PARTITION BY category
        ORDER BY total_revenue DESC
    ) AS rank_in_category


FROM product_revenue

ORDER BY
    category,
    rank_in_category""",conn)

,category,product_name,total_revenue,rank_in_category
0,Books,Body Fiction,27574.068214,1
1,Books,Explain Science,22196.889630,2
2,Books,Baby Comics,17968.155804,3
3,Books,Outside Biography,15114.036000,4
4,Books,Able Science,14079.612072,5
...,...,...,...,...
334,Home,Election Kitchen,62.016850,72
335,Home,Land Lighting,-260.960700,73
336,Home,Culture Decor,-697.969360,74
337,Home,Follow Storage,-2868.236010,75


__For each customer, calculate days between consecutive orders.(Flag customers with average gap > 30 days as "At Risk")__

In [27]:
pd.read_sql("""
WITH customer_orders AS (

    SELECT
        customer_id,
        order_date,

        LAG(order_date) OVER(
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_date

    FROM orders

),

order_gaps AS (

    SELECT
        customer_id,
        order_date,
        previous_order_date,

        CAST(
            julianday(order_date) -
            julianday(previous_order_date)
            AS INTEGER
        ) AS days_gap

    FROM customer_orders

),

customer_risk AS (

    SELECT
        customer_id,

        AVG(days_gap) AS average_gap,

        CASE
            WHEN AVG(days_gap) > 30
            THEN 'At Risk'
            ELSE 'Active'
        END AS customer_status

    FROM order_gaps

    GROUP BY customer_id

)

SELECT
    og.customer_id,
    og.order_date,
    og.previous_order_date,
    og.days_gap,
    cr.customer_status

FROM order_gaps og

JOIN customer_risk cr

ON og.customer_id = cr.customer_id

ORDER BY
    og.customer_id,
    og.order_date;

""", conn)

,customer_id,order_date,previous_order_date,days_gap,customer_status
0,8.0,2024-09-29 05:18:33,None,NaN,Active
1,17.0,18-06-2026,None,NaN,Active
2,21.0,2025-09-16 00:55:13,None,NaN,Active
3,22.0,2026-04-14 19:25:06,None,NaN,Active
4,29.0,2025-04-25 18:19:54,None,NaN,Active
...,...,...,...,...,...
341,4910.0,2025-06-21 00:48:55,None,NaN,Active
342,4912.0,2025-04-18 23:04:35,None,NaN,Active
343,4929.0,2025-09-30 13:47:55,None,NaN,Active
344,4937.0,2025-07-03 21:02:22,None,NaN,Active


- First, calculate monthly revenue per customer
- Then, categorize customers: 'High' (>10000), 'Medium' (5000-10000), 'Low' (<5000)
- Finally, show count of customers in each category per month

In [28]:
pd.read_sql("""WITH monthly_customer_revenue AS (

    SELECT
        o.customer_id,

        strftime('%Y-%m', o.order_date) AS month,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS monthly_revenue

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    GROUP BY
        o.customer_id,
        strftime('%Y-%m', o.order_date)

),

customer_category AS (

    SELECT
        customer_id,
        month,
        monthly_revenue,

        CASE

            WHEN monthly_revenue > 10000
            THEN 'High'

            WHEN monthly_revenue BETWEEN 5000 AND 10000
            THEN 'Medium'

            ELSE 'Low'

        END AS customer_segment

    FROM monthly_customer_revenue

)

SELECT

    month,

    customer_segment,

    COUNT(customer_id) AS customer_count

FROM customer_category

GROUP BY
    month,
    customer_segment

ORDER BY
    month,
    customer_segment;

""",conn)


,month,customer_segment,customer_count
0,None,High,3
1,None,Low,2
2,None,Medium,3
3,2024-07,Low,3
4,2024-08,High,3
...,...,...,...
62,2026-06,High,2
63,2026-06,Low,7
64,2026-06,Medium,1
65,2026-07,High,1


__Divide customers into 4 quartiles based on total lifetime value.__

In [29]:
query = """
WITH customer_ltv AS (

    SELECT
        o.customer_id,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS total_value

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    GROUP BY
        o.customer_id

),

customer_quartiles AS (

    SELECT

        customer_id,
        total_value,

        NTILE(4) OVER(
            ORDER BY total_value DESC
        ) AS quartile

    FROM customer_ltv

)

SELECT

    customer_id,
    total_value,
    quartile,

    CASE

        WHEN quartile = 1
        THEN 'Platinum'

        WHEN quartile = 2
        THEN 'Gold'

        WHEN quartile = 3
        THEN 'Silver'

        WHEN quartile = 4
        THEN 'Bronze'

    END AS quartile_label


FROM customer_quartiles

ORDER BY
    quartile,
    total_value DESC;

"""

result = pd.read_sql(query, conn)

result

,customer_id,total_value,quartile,quartile_label
0,NaN,49651.556893,1,Platinum
1,2693.0,30227.572410,1,Platinum
2,2532.0,23145.391494,1,Platinum
3,2117.0,22196.889630,1,Platinum
4,2826.0,21513.638586,1,Platinum
...,...,...,...,...
200,163.0,60.885059,4,Bronze
201,4353.0,16.850262,4,Bronze
202,1652.0,1.050380,4,Bronze
203,4693.0,-1093.495590,4,Bronze


__Compare each month's revenue with same month previous year.__

In [30]:
pd.read_sql("""WITH monthly_revenue AS (

    SELECT
        CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
        CAST(strftime('%m', o.order_date) AS INTEGER) AS month,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    GROUP BY
        strftime('%Y', o.order_date),
        strftime('%m', o.order_date)

)

SELECT

    curr.year,
    curr.month,
    ROUND(curr.revenue, 2) AS revenue,

    ROUND(prev.revenue, 2) AS prev_year_revenue,

    CASE
        WHEN prev.revenue IS NULL
             OR prev.revenue = 0
        THEN NULL

        ELSE ROUND(
            (
                (curr.revenue - prev.revenue)
                * 100.0
            ) / prev.revenue,
            2
        )
    END AS yoy_growth_percent

FROM monthly_revenue curr

LEFT JOIN monthly_revenue prev

    ON curr.month = prev.month
   AND curr.year = prev.year + 1

ORDER BY
    curr.year,
    curr.month""",conn)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,NaN,NaN,80420.16,NaN,NaN
1,2024.0,7.0,9404.01,NaN,NaN
2,2024.0,8.0,42256.00,NaN,NaN
3,2024.0,9.0,34616.75,NaN,NaN
4,2024.0,10.0,20648.06,NaN,NaN
5,2024.0,11.0,50837.69,NaN,NaN
6,2024.0,12.0,66633.10,NaN,NaN
7,2025.0,1.0,50909.64,NaN,NaN
8,2025.0,2.0,41712.67,NaN,NaN
9,2025.0,3.0,40758.90,NaN,NaN


__For each customer, show their first purchased category and most recent purchased category.
Flag if they are different (category_shift = 'Yes'/'No')__

In [31]:
pd.read_sql("""WITH customer_categories AS (

    SELECT
        o.customer_id,
        p.category,
        o.order_date,

        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date ASC
        ) AS first_rn,

        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date DESC
        ) AS last_rn

    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    JOIN products p
        ON oi.product_id = p.product_id
),

first_category AS (

    SELECT
        customer_id,
        category AS first_purchased_category
    FROM customer_categories
    WHERE first_rn = 1
),

last_category AS (

    SELECT
        customer_id,
        category AS most_recent_purchased_category
    FROM customer_categories
    WHERE last_rn = 1
)

SELECT
    f.customer_id,
    f.first_purchased_category,
    l.most_recent_purchased_category,

    CASE
        WHEN f.first_purchased_category =
             l.most_recent_purchased_category
        THEN 'No'
        ELSE 'Yes'
    END AS category_shift

FROM first_category f
JOIN last_category l
    ON f.customer_id = l.customer_id

ORDER BY f.customer_id""",conn)

,customer_id,first_purchased_category,most_recent_purchased_category,category_shift
0,8.0,Clothing,Clothing,No
1,17.0,Electronics,Electronics,No
2,22.0,Home,Home,No
3,80.0,Clothing,Clothing,No
4,84.0,Home,Home,No
...,...,...,...,...
199,4877.0,Electronics,Clothing,Yes
200,4912.0,Electronics,Electronics,No
201,4929.0,Home,Home,No
202,4937.0,Home,Home,No


__Calculate what percentage of total revenue comes from top N% of customers.__

In [32]:
pd.read_sql("""WITH customer_revenue AS (

    SELECT
        o.customer_id,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue

    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id

    GROUP BY o.customer_id
),

ranked_customers AS (

    SELECT
        customer_id,
        revenue,

        SUM(revenue) OVER (
            ORDER BY revenue DESC
        ) AS cumulative_revenue,

        SUM(revenue) OVER () AS total_revenue

    FROM customer_revenue
)

SELECT
    customer_id,

    ROUND(revenue, 2) AS revenue,

    ROUND(cumulative_revenue, 2) AS cumulative_revenue,

    ROUND(
        cumulative_revenue * 100.0 /
        total_revenue,
        2
    ) AS cumulative_percent

FROM ranked_customers

ORDER BY revenue DESC""",conn)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,NaN,49651.56,49651.56,4.11
1,2693.0,30227.57,79879.13,6.60
2,2532.0,23145.39,103024.52,8.52
3,2117.0,22196.89,125221.41,10.35
4,2826.0,21513.64,146735.05,12.13
...,...,...,...,...
200,163.0,60.89,1211872.29,100.20
201,4353.0,16.85,1211889.14,100.21
202,1652.0,1.05,1211890.19,100.21
203,4693.0,-1093.50,1210796.69,100.12


__Group customers by their registration month (cohort).__

In [33]:
pd.read_sql("""WITH customer_cohorts AS (

    SELECT
        customer_id,
        date(registration_date, 'start of month') AS cohort_month
    FROM customers

),

customer_orders AS (

    SELECT DISTINCT
        c.customer_id,
        c.cohort_month,

        (
            (CAST(strftime('%Y', o.order_date) AS INTEGER) -
             CAST(strftime('%Y', c.cohort_month) AS INTEGER)) * 12

            +

            (CAST(strftime('%m', o.order_date) AS INTEGER) -
             CAST(strftime('%m', c.cohort_month) AS INTEGER))

        ) AS month_number

    FROM customer_cohorts c
    JOIN orders o
        ON c.customer_id = o.customer_id

),

cohort_size AS (

    SELECT
        cohort_month,
        COUNT(*) AS customers_registered
    FROM customer_cohorts
    GROUP BY cohort_month

),

retention AS (

    SELECT
        cohort_month,

        COUNT(DISTINCT CASE WHEN month_number = 0 THEN customer_id END) AS month_0,

        COUNT(DISTINCT CASE WHEN month_number = 1 THEN customer_id END) AS month_1,

        COUNT(DISTINCT CASE WHEN month_number = 2 THEN customer_id END) AS month_2,

        COUNT(DISTINCT CASE WHEN month_number = 3 THEN customer_id END) AS month_3

    FROM customer_orders

    WHERE month_number BETWEEN 0 AND 3

    GROUP BY cohort_month

)

SELECT

    r.cohort_month,

    c.customers_registered,

    r.month_0,
    ROUND(100.0 * r.month_0 / c.customers_registered, 2) AS retention_m0,

    r.month_1,
    ROUND(100.0 * r.month_1 / c.customers_registered, 2) AS retention_m1,

    r.month_2,
    ROUND(100.0 * r.month_2 / c.customers_registered, 2) AS retention_m2,

    r.month_3,
    ROUND(100.0 * r.month_3 / c.customers_registered, 2) AS retention_m3

FROM retention r

JOIN cohort_size c
    ON r.cohort_month = c.cohort_month

ORDER BY r.cohort_month""",conn)

,cohort_month,customers_registered,month_0,retention_m0,month_1,retention_m1,month_2,retention_m2,month_3,retention_m3
0,2024-05-01,88,0,0.00,0,0.00,0,0.00,1,1.14
1,2024-07-01,80,0,0.00,1,1.25,0,0.00,1,1.25
2,2024-09-01,86,0,0.00,0,0.00,0,0.00,1,1.16
3,2024-10-01,72,0,0.00,0,0.00,0,0.00,1,1.39
4,2024-11-01,77,0,0.00,0,0.00,0,0.00,1,1.30
5,2025-01-01,89,0,0.00,1,1.12,0,0.00,0,0.00
6,2025-03-01,74,0,0.00,0,0.00,1,1.35,0,0.00
7,2025-04-01,73,0,0.00,0,0.00,3,4.11,0,0.00
8,2025-05-01,102,0,0.00,1,0.98,1,0.98,0,0.00
9,2025-06-01,86,0,0.00,0,0.00,0,0.00,1,1.16


__Find products frequently bought together.__

In [34]:
pd.read_sql("""SELECT

    p1.product_name AS product_a,
    p2.product_name AS product_b,

    COUNT(*) AS times_bought_together

FROM order_items oi1

JOIN order_items oi2
    ON oi1.order_id = oi2.order_id
   AND oi1.product_id < oi2.product_id

JOIN products p1
    ON oi1.product_id = p1.product_id

JOIN products p2
    ON oi2.product_id = p2.product_id

GROUP BY
    p1.product_name,
    p2.product_name

ORDER BY
    times_bought_together DESC,
    product_a,
    product_b""",conn)

,product_a,product_b,times_bought_together
0,Base Lighting,Financial Laptop,1
1,Ability Storage,So Tablet,1
2,Able Laptop,Commercial Biography,1
3,Able Laptop,Follow Storage,1
4,Able Science,Own Jeans,1
...,...,...,...
225,Young Decor,Offer Mobile,1
226,Young Decor,Song Jacket,1
227,Young T-Shirt,Eight Headphones,1
228,Young T-Shirt,Fact Camera,1


## Step 4 : Integrated CLI 

In [35]:

def get_previous_period(start_date, end_date):
    days = (end_date - start_date).days + 1

    prev_end = start_date - timedelta(days=1)
    prev_start = prev_end - timedelta(days=days - 1)

    return prev_start, prev_end


def pct_change(current, previous):
    if previous == 0:
        return None
    return round(((current - previous) / previous) * 100, 2)


def get_summary(cursor, start_date, end_date):

    query = """
    SELECT
        COUNT(DISTINCT o.order_id) AS total_orders,

        ROUND(
            COALESCE(
                SUM(
                    oi.quantity *
                    oi.unit_price *
                    (1 - oi.discount_percent / 100.0)
                ),
                0
            ),
            2
        ) AS revenue,

        COUNT(DISTINCT o.customer_id) AS unique_customers

    FROM orders o
    LEFT JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE date(o.order_date)
          BETWEEN date(?) AND date(?)
    """

    cursor.execute(query, (start_date, end_date))

    return cursor.fetchone()


def get_top_products(cursor, start_date, end_date):

    query = """
    SELECT
        p.product_name,
        SUM(oi.quantity) AS units_sold

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE date(o.order_date)
          BETWEEN date(?) AND date(?)

    GROUP BY p.product_id, p.product_name

    ORDER BY units_sold DESC

    LIMIT 3
    """

    cursor.execute(query, (start_date, end_date))

    return cursor.fetchall()


def main():

    print("\n==Order Analytics Report ==\n")

    report_type = input("Report Type (daily/weekly/monthly): ").strip().lower()

    if report_type not in ("daily", "weekly", "monthly"):
        print("Invalid report type")
        return

    start_date = input("Start Date (YYYY-MM-DD): ").strip()

    end_date = input("End Date (YYYY-MM-DD): ").strip()

    try:
        start_dt = datetime.strptime(
            start_date,
            "%Y-%m-%d"
        )

        end_dt = datetime.strptime(
            end_date,
            "%Y-%m-%d"
        )

    except ValueError:
        print("Invalid date format")
        return

    prev_start, prev_end = get_previous_period(
        start_dt,
        end_dt
    )

    conn = sqlite3.connect("ecommerce_analysis.db")
    cursor = conn.cursor()

    current = get_summary(
        cursor,
        start_date,
        end_date
    )

    previous = get_summary(
        cursor,
        prev_start.strftime("%Y-%m-%d"),
        prev_end.strftime("%Y-%m-%d")
    )

    top_products = get_top_products(
        cursor,
        start_date,
        end_date
    )

    current_orders = current[0] or 0
    current_revenue = current[1] or 0
    current_customers = current[2] or 0

    prev_orders = previous[0] or 0
    prev_revenue = previous[1] or 0
    prev_customers = previous[2] or 0

    print("REPORT SUMMARY")
    print("====")

    print(
        f"Period: {start_date} to {end_date}" )

    print(
        f"Report Type: {report_type.title()}")

    print("\nMetrics")

    print(
        f"Total Orders: {current_orders}"
    )

    print(
        f"Revenue: {current_revenue:.2f}"
    )

    print(
        f"Unique Customers: {current_customers}"
    )

    print("\nPrevious Period Comparison")

    print(
        f"Orders Change: "
        f"{pct_change(current_orders, prev_orders)}%"
    )

    print(
        f"Revenue Change: "
        f"{pct_change(current_revenue, prev_revenue)}%"
    )

    print(
        f"Customer Change: "
        f"{pct_change(current_customers, prev_customers)}%"
    )

    print("\nTop 3 Products")

    for rank, row in enumerate(
        top_products,
        start=1
    ):
        print(
            f"{rank}. {row[0]} "
            f"(Units Sold: {row[1]})"
        )

if __name__ == "__main__":
    main()


==Order Analytics Report ==



Report Type (daily/weekly/monthly):  daily
Start Date (YYYY-MM-DD):  2026-07-08
End Date (YYYY-MM-DD):  2026-07-09


REPORT SUMMARY
====
Period: 2026-07-08 to 2026-07-09
Report Type: Daily

Metrics
Total Orders: 1
Revenue: 0.00
Unique Customers: 1

Previous Period Comparison
Orders Change: 0.0%
Revenue Change: None%
Customer Change: 0.0%

Top 3 Products


## Step 5: Edge Cases 

In [36]:
conn.close()

In [37]:

def test_orphan_order_items(conn):
    """
    order_items.order_id exists but orders.order_id does not.
    """

    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM order_items oi
    LEFT JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
    """)

    orphan_count = cursor.fetchone()[0]

    assert orphan_count == 0, (
        f"Found {orphan_count} orphan order_items"
    )

    print("PASS: No orphan order_items found")


def test_discount_greater_than_100(conn):
    """
    discount_percent should never exceed 100.
    """

    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM order_items
    WHERE discount_percent > 100
    """)

    invalid_count = cursor.fetchone()[0]

    assert invalid_count == 0, (
        f"Found {invalid_count} rows with discount > 100%"
    )

    print("PASS: All discounts are valid")


def test_zero_quantity(conn):
    """
    quantity should be greater than zero.
    """

    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM order_items
    WHERE quantity = 0
    """)

    zero_qty_count = cursor.fetchone()[0]

    assert zero_qty_count == 0, (
        f"Found {zero_qty_count} rows with quantity = 0"
    )

    print("PASS: No zero quantity rows found")


def test_future_order_dates(conn):
    """
    order_date should not be in the future.
    """

    today = date.today().isoformat()

    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM orders
    WHERE date(order_date) > date(?)
    """, (today,))

    future_count = cursor.fetchone()[0]

    assert future_count == 0, (
        f"Found {future_count} future orders"
    )

    print("PASS: No future-dated orders found")


def run_all_tests(db_path):

    conn = sqlite3.connect(db_path)

    try:

        print("\nRunning Data Quality Tests...\n")

        test_orphan_order_items(conn)

        test_discount_greater_than_100(conn)

        test_zero_quantity(conn)

        test_future_order_dates(conn)

        print("\nALL TESTS PASSED")

    except Exception as Invalid:
            print(f"\nTest failed: {Invalid}")


In [38]:
conn.close()

# Summary
# E-Commerce Order Analytics System

## Project Summary

This project simulates a real-world data engineering workflow for an e-commerce company. The objective is to process raw order data from multiple sources, identify and resolve data quality issues, transform the data into an analytics-ready format, and generate business insights through SQL and Python.

The project includes:

* Generation of realistic e-commerce datasets using Faker
* Data cleaning and validation
* Duplicate and null value handling
* Business rule validation and referential integrity checks
* Loading cleaned data into SQLite
* Advanced SQL analytics using joins, CTEs, and window functions
* Automated report generation through Python and SQL integration
* Edge case testing and error handling
* Logging and modular ETL design

### Dataset Tables

* **customers** – Customer information and registration details
* **products** – Product catalog and category information
* **orders** – Order-level transaction records
* **order_items** – Product-level purchase details

### Key Business Analysis 

* Revenue and sales trends
* Customer Lifetime Value (CLV)
* Cohort retention analysis
* Frequently bought together products
* Year-over-Year growth analysis
* Revenue concentration (Pareto analysis)
* Product and category performance

### Technologies Used

* Python
     Pandas
     Faker
     Logging
     os
     re
* SQL
* Jupyter Notebook
